In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [2]:
# 欄位處理 Column processing
def check_data_format(df_data, label_value):
    
    # 生活習慣&疾病史，空值刪除
    df_data = df_data.dropna(subset=["AGE","DRK","SMK_EXPERIENCE","NUT_EXPERIENCE","SPO_HABIT","ORTHOPEDICS_ARTICULUS","OSTEOPOROSIS","ARTHRITIS","GOUT","LUNG_RESPIRATORY","ASTHMA","EMPHYSEMA_BRONCHITIS","HEART_BLOOD_TUBE","VALVE_HEART_DIS","CORONARY_ARTERY_DIS","ARRHYTHMIA","CARDIOMYOPATHY","CONGENITAL_HEART_DIS","OTHER_HEART_DIS","HYPERLIPIDEMIA","HYPERTENSION","APOPLEXIA","DIABETES","ALIMENTARY_CANAL","PEPTIC_ULCER","GASTROESOPHAGEA_REFLUX","IRRITABLE_BOWEL_SND","PSYCHOSIS","DEPRESSION","MANIC_DEPRESSION","POSTPARTUM_DEPRESSION","OBSESSIVE_COMPULSIVE_DIS","ALCOHOLISM_DRUG_ABUSE","SCHIZOPHRENIA","NERVOUS_SYSTEM","EPILEPSY","HEMICRANIA","MULTIPLE_SCLEROSIS","PARKISON","DEMENTIA","LIVER_GALL_STONE","KIDNEY_STONE","RENAL_FAILURE","VERTIGO","CARCINOMA_IN_SITU","LIVER_CA","LUNG_CA","BREAST_CA","GASTRIC_CA","COLORECTAL_CA","NASOPHARYNGEAL_CA","PROSTATE_CA","OTHER_CA","CATARACT","GLAUCOMA","XEROPHTHALMIA","RENTINAL_DETACHMENT","FLOATERS","BLIND","COLOR_BLIND"])
    #print("刪除生活習慣及疾病史空值後剩餘:" + str(df_data.shape))
    
    # 身體及血尿液量測數值，空值刪除
    df_data = df_data.dropna(subset=["BODY_HEIGHT","BODY_WEIGHT","BMI","BODY_WAISTLINE","BODY_BUTTOCKS","WHR","SIT_1_SYSTOLIC_PRESSURE","SIT_1_DIASTOLIC_PRESSURE","SIT_2_SYSTOLIC_PRESSURE","SIT_2_DIASTOLIC_PRESSURE","SIT_1_HEARTBEAT_SPEED","SIT_2_HEARTBEAT_SPEED","BONE_EXAM_RESULT","T_SCORE","Z_SCORE","BODY_FAT_RATE","RBC","WBC","PLATELET","HB","HCT","HBA1C","ANTI_HCV_AB_1","HBSAG_1","HBEAG_1","ANTI_HBS_AB_1","ANTI_HBC_AB_1","FASTING_GLUCOSE","T_CHO","TG","HDL_C","LDL_C","T_BILIRUBIN","ALBUMIN","SGOT","SGPT","GAMMA_GT","AFP","BUN","CREATININE","URIC_ACID","MICROALB"])
    #print("刪除身體及血尿液量測數值空值後剩餘:" + str(df_data.shape))

    # 女性相關問題之類別資料，空值刪除
    df_data = df_data.dropna(subset=["MENSTRURATION_REGULAR","MENSTRURATION_CURR","PREGNANCY","SEVERE_DYSMENORRHEA","MYOMA","OVARIAN_CYST","ENDOMETRIOSIS","CERVICAL_POLYP","UTERINE_CA","CERVICAL_CA","OVARIAN_CA"])
    #print("刪除女性相關問題之類別資料空值後剩餘:" + str(df_data.shape))

    # 女性相關問題:初經年齡為空值，使用眾數取代
    #MENARCHE_cl = ctl_df.groupby('MENARCHE')
    #print(MENARCHE_cl.size())
    #'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead
    df_data['MENARCHE'] = df_data['MENARCHE'].fillna(df_data['MENARCHE'].mode()[0])
    #df.fillna({'MENARCHE': df_data['MENARCHE'].mode()[0]}, inplace=True)
    #print("初經年齡眾數:" + str(df_data['MENARCHE'].mode()[0]))

    # 女性相關問題:停經年齡，已停經且停經年齡為空值，使用眾數取代
    #MENOPAUSE_cl = ctl_df.groupby('MENOPAUSE')
    #print(MENOPAUSE_cl.size())
    MENOPAUSE_mode = df_data['MENOPAUSE'].mode()[0]
    df_data['MENOPAUSE'] = df_data.apply(
        lambda row: MENOPAUSE_mode if row['MENSTRURATION_CURR'] == '1' and pd.isna(row['MENOPAUSE']) else row['MENOPAUSE'],
        axis=1
    )
    #print("停經年齡眾數:" + str(MENOPAUSE_mode))

    # 女性相關問題:新增欄位"過去及至今月經期間"，(1)已停經:停經年齡-初經年齡(2)未停經:現在年齡-初經年齡
    df_data.insert(len(df_data.columns), "MC_DURATION",True)
    df_data['MC_DURATION'] = df_data.apply(
        lambda row: int(row['MENOPAUSE'])-int(row['MENARCHE']) if row['MENSTRURATION_CURR'] == '1' else int(row['AGE'])-int(row['MENARCHE']),
        axis=1
    )

    # 女性相關問題:未曾懷孕的樣本中，"是否曾經生產過"、"生產次數"、"是否曾經流產過"、"是否曾經餵過母乳"會顯示空值，需補值
    df_data['BIRTH'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH']) else row['BIRTH'],
        axis=1
    )
    df_data['BIRTH_TIMES'] = df_data.apply(
        lambda row: '0' if row['BIRTH'] == '1' and pd.isna(row['BIRTH_TIMES']) else row['BIRTH_TIMES'],
        axis=1
    )

    df_data['ABORTION'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['ABORTION']) else row['ABORTION'],
        axis=1
    )
    df_data['BREAST_MILK'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BREAST_MILK']) else row['BREAST_MILK'],
        axis=1
    )

    # 女性相關問題:"是否曾經生產過"補值後依然有空值，刪除樣本(例如:曾經懷孕，"是否曾經生產過"卻顯示空值)
    df_data = df_data.dropna(subset=["BIRTH","BIRTH_TIMES","ABORTION","BREAST_MILK"])

    # 女性相關問題:新增欄位"是否高齡產"
    # 未曾懷孕的樣本中，"最後生產年齡"為空值，補值[-1]，協助判斷未填入"最後生產年齡"的樣本以便刪除
    df_data['BIRTH_AGE_FINAL'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH_AGE_FINAL']) else row['BIRTH_AGE_FINAL'],
        axis=1
    )
    df_data = df_data.dropna(subset=["BIRTH_AGE_FINAL"])
    #print("刪除女性相關問題無法補值之樣本後剩餘:" + str(df_data.shape))

    # 依"最後生產年齡"給值，>=35是高齡產，<35則否
    df_data.insert(69, "ELDERL_PRIMIGRAVIDA",True)
    df_data['ELDERL_PRIMIGRAVIDA'] = df_data.apply(
        lambda row: '1' if int(row['BIRTH_AGE_FINAL']) < 35 else '2',
        axis=1
    )

    # 身體及血尿液量測數值，特殊值處理
    # 平均收縮壓、舒張壓、心跳
    df_data.insert(91, "SIT_SYSTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_SYSTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_SYSTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_SYSTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(92, "SIT_DIASTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_DIASTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_DIASTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_DIASTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(93, "SIT_HEARTBEAT_SPEED_AVERAGE",True)
    df_data['SIT_HEARTBEAT_SPEED_AVERAGE'] = (df_data['SIT_1_HEARTBEAT_SPEED'].astype(float) + df_data['SIT_2_HEARTBEAT_SPEED'].astype(float))/2

    # 肺功能測量結果，文字代換成數字
    df_data['INTERPRETATION'] = df_data['INTERPRETATION'].map({'Normal Spirometry':1, 'Restrictive Impairment':2, 'Obstructive Impairment':3, 'Mixed Impairment':4})

    # 醣化血色素值，<3.8用0取代
    df_data['HBA1C'] = df_data['HBA1C'].replace(to_replace={'<3.8': 0})

    # C型肝炎抗體、B型肝炎表面抗原、B型肝炎e抗原、B型肝炎表面抗體、B型肝炎核心抗體，文字代換成數字
    df_data['ANTI_HCV_AB_1'] = df_data['ANTI_HCV_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBSAG_1'] = df_data['HBSAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBEAG_1'] = df_data['HBEAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBS_AB_1'] = df_data['ANTI_HBS_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBC_AB_1'] = df_data['ANTI_HBC_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})

    # 血清麩胺酸苯醋酸轉氨基酶，<6用0取代
    df_data['SGOT'] = df_data['SGOT'].replace(to_replace={'<6': 0})
    
    # 血清麩胺酸: 0}，<5用0取代
    df_data['SGPT'] = df_data['SGPT'].replace(to_replace={'<5': 0})

    # γ －麩胺醯轉移酶，<3.3用0取代
    df_data['GAMMA_GT'] = df_data['GAMMA_GT'].replace(to_replace={'<3.3': 0})

    # 甲型胎兒血清蛋白，<2用0取代
    df_data['AFP'] = df_data['AFP'].replace(to_replace={'< 2.00': 0, '<2': 0, '<2.0': 0, '< 2.0': 0})
    
    # 尿酸，<0.6用0取代
    df_data['URIC_ACID'] = df_data['URIC_ACID'].replace(to_replace={'<0.6': 0})

    # 尿中微蛋白，<2.09用0取代
    df_data['MICROALB'] = df_data['MICROALB'].replace(to_replace={'<2.09': 0})
    
    # 新增label
    df_data.insert(128, "OS",True)
    df_data['OS'] = label_value
    
    #print("最終樣本數:" + str(df_data.shape))

    return df_data

In [3]:
df_n = pd.read_csv ('./data/oste_normal(20240409).csv', low_memory=False)
ctl_df = df_n[1:]
print("無骨鬆原始資料:" + str(ctl_df.shape))
#ctl_df_filter = ctl_df[pd.to_numeric(ctl_df['T_SCORE']) > 2.3]
#print(ctl_df_filter.shape)

re_ctl_df = check_data_format(ctl_df, '0')
print("無骨鬆欄位處理後資料:" + str(re_ctl_df.shape))

df_oa = pd.read_csv ('./data/oste_osteopenia(20240409).csv', low_memory=False)
oa_df = df_oa[1:]
print("輕微骨鬆原始資料:" + str(oa_df.shape))
#oa_df_filter = oa_df[(pd.to_numeric(oa_df['T_SCORE']) > -2.0) & (pd.to_numeric(oa_df['T_SCORE']) < -1.5)]
#print(oa_df_filter.shape)

re_oa_df = check_data_format(oa_df, '2')
print("輕微骨鬆欄位處理後資料:" + str(re_oa_df.shape))

df_os = pd.read_csv ('./data/oste_osteoporosis(20240409).csv', low_memory=False)
os_df = df_os[1:]
print("骨鬆原始資料:" + str(os_df.shape))

re_os_df = check_data_format(os_df, '1')
print("骨鬆欄位處理後資料:" + str(re_os_df.shape))

無骨鬆原始資料:(67283, 123)
無骨鬆欄位處理後資料:(59265, 129)
輕微骨鬆原始資料:(26366, 123)
輕微骨鬆欄位處理後資料:(23027, 129)
骨鬆原始資料:(7910, 123)
骨鬆欄位處理後資料:(6703, 129)


In [4]:
# 抽取有snp資訊的樣本
snp_ctl_df = re_ctl_df.dropna(subset=["TWB2_ID"])
print(str(snp_ctl_df.shape))

snp_oa_df = re_oa_df.dropna(subset=["TWB2_ID"])
print(str(snp_oa_df.shape))

snp_os_df = re_os_df.dropna(subset=["TWB2_ID"])
print(str(snp_os_df.shape))

(43232, 129)
(17352, 129)
(4978, 129)


In [5]:
# 取得候選SNP集合可對應到TWB資料庫的SNP_ID資訊
map_file = './data/tpmi_snp.map'
snp_name_list = []
map_f = open(map_file, 'r')
for line in map_f:
    line = line.strip()
    line = line.split('\t')
    snp_id = line[1]
    snp_name_list.append(snp_id)
print("SNP List:" + str(len(snp_name_list)))
print(snp_name_list)

SNP List:9
['rs35645182', 'rs34782958', 'rs2198891', 'rs190752175', 'rs4712614', 'rs35257346', 'rs1871859', 'rs635965', 'rs867023908']


In [6]:
# 新增SNP欄位到三組dataframe
for rs_id in snp_name_list:
    snp_os_df.insert(len(snp_os_df.columns), rs_id, None)
    snp_oa_df.insert(len(snp_oa_df.columns), rs_id, None)
    snp_ctl_df.insert(len(snp_ctl_df.columns), rs_id, None)

In [7]:
# 將TWB2_ID欄位設為index
snp_os_df = snp_os_df.set_index('TWB2_ID')
snp_oa_df = snp_oa_df.set_index('TWB2_ID')
snp_ctl_df = snp_ctl_df.set_index('TWB2_ID')

In [8]:
from collections import deque
def asign_snp_allele(ped_file_name, snp_df):
    ped_f = open(ped_file_name, 'r')
    for line in ped_f:
        line = line.strip()
        line = line.split('\t')
        idx = line[0]
        #print(idx)
        snp_count = len(line[6:])/2
        snp_list = line[6:]
        #print(snp_list)
        queue = deque(snp_list)
        for i in range(int(len(snp_list)/2)):
            chr_1 = queue.popleft()
            chr_2 = queue.popleft()
            #print(chr_1)
            #print(chr_2)
            snp_name = snp_name_list[i]
            #print(snp_name)
            
            #snp_df.loc[idx, snp_name] = 
            if idx in snp_df.index:
                snp_df.at[idx, snp_name] = chr_1+'_'+chr_2
                #snp_df.at[idx, snp_name] = index_nt_biallele_code[nt_biallele_code[t_chr]]
    return snp_df

In [9]:
snp_os_df = asign_snp_allele('./data/tpmi_osteoporosis.ped', snp_os_df)
#print(snp_os_df)
snp_oa_df = asign_snp_allele('./data/tpmi_osteopenia.ped', snp_oa_df)
#print(snp_oa_df)
snp_ctl_df = asign_snp_allele('./data/tpmi_normal.ped', snp_ctl_df)
#print(snp_ctl_df)

In [10]:
# 讀取PRS file
snp_prs_df = pd.read_csv('./prs/tpmi_osteoporosis_prs.txt', sep='\t')
#print(snp_prs_df)
snp_prs_list = snp_prs_df['source'].unique()
print(snp_prs_list)

<StringArray>
['TPMI_Osteoporosis_osteopenia and pathological fracture',
                                      'TPMI_Osteoporosis',
                                  'TPMI_Osteoporosis NOS',
                               'TPMI_Senile osteoporosis']
Length: 4, dtype: str


In [11]:
# 新增PRS欄位到三組dataframe
for rs_group in snp_prs_list:    
    snp_os_df.insert(len(snp_os_df.columns), rs_group, None)
    snp_oa_df.insert(len(snp_oa_df.columns), rs_group, None)
    snp_ctl_df.insert(len(snp_ctl_df.columns), rs_group, None)

In [12]:
def alt_conut(genotype, alt):
    if pd.isna(genotype):
        return 0
    a1, a2 = genotype.split("_")
    return (a1 == alt) + (a2 == alt)
    
def col_prs(snp_df):
    for source, gdf in snp_prs_df.groupby("source"):
    #for source, gdf in snp_df.groupby("source"):
        #print(source)
        #print(gdf)
        prs = pd.Series(0.0, index=snp_df.index)

        for _, row in gdf.iterrows():
            rs = row["rs_id"]
            alt = row["ALT_allele(A1)"]
            es = row["effect_size"]
            #print(rs)
            #print(alt)
            #print(es)
            if rs not in snp_df.columns:
                #print("no_rsid")
                #print(rs)
                continue
            #print(snp_df[rs])
            count = snp_df[rs].apply(alt_conut, alt=alt)
            #print(count)
            prs += count * es
            #print(prs)
        #print(source)
        snp_df[source] = prs
        #print(snp_df[source])
    return snp_df

In [13]:
snp_os_df = col_prs(snp_os_df)
print(str(snp_os_df.shape))
snp_oa_df = col_prs(snp_oa_df)
print(str(snp_oa_df.shape))
snp_ctl_df = col_prs(snp_ctl_df)
print(str(snp_ctl_df.shape))

(4978, 141)
(17352, 141)
(43232, 141)


In [14]:
# 新增欄位到三組dataframe
for rs_id in snp_name_list:
    snp_os_df.insert(len(snp_os_df.columns), rs_id + "_cate", None)
    snp_oa_df.insert(len(snp_oa_df.columns), rs_id + "_cate", None)
    snp_ctl_df.insert(len(snp_ctl_df.columns), rs_id + "_cate", None)

In [15]:
# 讀取SNP編碼檔案
import json
nt_biallele_code = json.load(open('./snp2vec/snp_vocab.json', 'r',encoding='utf-8'))
index_nt_biallele_code = json.load(open('./snp2vec/vocab_nucleotide_to_index.json', 'r',encoding='utf-8'))

# TWB SNP資料轉換使之與編碼檔一致
def trans_snp_chr(word):
    if(len(word) > 1):
        if(word[0] == '0'):
            word[0] = 'N'
        word = word[0]+'I'
    else:
        if(word == '-'):
            word = 'DEL'
        if(word == '0'):
            word = 'N'
    return word

# 將ATCG組合轉換成數值類別
from collections import deque
def asign_snp_label(ped_file_name, snp_df):
    ped_f = open(ped_file_name, 'r')
    for line in ped_f:
        line = line.strip()
        line = line.split('\t')
        idx = line[0]
        #print(idx)
        snp_count = len(line[6:])/2
        snp_list = line[6:]
        #print(snp_list)
        queue = deque(snp_list)
        for i in range(int(len(snp_list)/2)):
            chr_1 = queue.popleft()
            chr_2 = queue.popleft()
            #print(chr_1)
            #print(chr_2)
            snp_name = snp_name_list[i]
            #print(snp_name)
            if(chr_1 == chr_2):
                t_chr = trans_snp_chr(chr_1)
                #print(nt_biallele_code[t_chr])
                #print(index_nt_biallele_code[nt_biallele_code[t_chr]])
                #if idx in snp_df.index:
                    #snp_df.at[idx, snp_name] = index_nt_biallele_code[nt_biallele_code[t_chr]]
                    #snp_df.at[idx, snp_name] = nt_biallele_code[t_chr]
            else:
                t_chr_1 = trans_snp_chr(chr_1)
                t_chr_2 = trans_snp_chr(chr_2)
                if(ord(t_chr_1[0]) > ord(t_chr_2[0])):
                    #print(t_chr_1)
                    #print(t_chr_2)
                    temp = t_chr_1
                    t_chr_1 = t_chr_2
                    t_chr_2 = temp
                if(ord(t_chr_1[0]) == ord(t_chr_2[0])):
                    if(len(t_chr_1) > len(t_chr_2)):
                        temp = t_chr_1
                        t_chr_1 = t_chr_2
                        t_chr_2 = temp
                t_chr = t_chr_1+'_'+t_chr_2
                #print(nt_biallele_code[t_chr_1+'_'+t_chr_2])
                #print(index_nt_biallele_code[nt_biallele_code[t_chr_1+'_'+t_chr_2]])
            if idx in snp_df.index:
                snp_df.at[idx, snp_name + '_cate'] = index_nt_biallele_code[nt_biallele_code[t_chr]]
                #snp_df.at[idx, snp_name] = index_nt_biallele_code[nt_biallele_code[t_chr]]
                #snp_df.at[idx, snp_name] = nt_biallele_code[t_chr_1+'_'+t_chr_2]
    return snp_df

In [16]:
snp_os_df = asign_snp_label('./data/tpmi_osteoporosis.ped', snp_os_df)
print(str(snp_os_df.shape))
snp_oa_df = asign_snp_label('./data/tpmi_osteopenia.ped', snp_oa_df)
print(str(snp_oa_df.shape))
snp_ctl_df = asign_snp_label('./data/tpmi_normal.ped', snp_ctl_df)
print(str(snp_ctl_df.shape))

(4978, 150)
(17352, 150)
(43232, 150)


In [17]:
#合併所有分類資料
merge_all = pd.concat([snp_os_df, snp_ctl_df])
merge_all = pd.concat([merge_all, snp_oa_df])

print(merge_all.shape)

(65562, 150)


In [18]:
#切分train data & test data
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    merge_all,
    test_size=0.2,
    random_state=42
)

print("train shape:", train_df.shape)
print(train_df['OS'].value_counts())
print("test shape:", test_df.shape)
print(test_df['OS'].value_counts())

train shape: (52449, 150)
OS
0    34544
2    13898
1     4007
Name: count, dtype: int64
test shape: (13113, 150)
OS
0    8688
2    3454
1     971
Name: count, dtype: int64


In [19]:
def ectract_cate(input_df):
    #feature select
    re_cate_df = input_df.filter(items=["DRK","SMK_EXPERIENCE","NUT_EXPERIENCE","SPO_HABIT","ORTHOPEDICS_ARTICULUS","OSTEOPOROSIS","ARTHRITIS","GOUT","LUNG_RESPIRATORY","ASTHMA","EMPHYSEMA_BRONCHITIS","HEART_BLOOD_TUBE","VALVE_HEART_DIS","CORONARY_ARTERY_DIS","ARRHYTHMIA","CARDIOMYOPATHY","CONGENITAL_HEART_DIS","OTHER_HEART_DIS","HYPERLIPIDEMIA","HYPERTENSION","APOPLEXIA","DIABETES","ALIMENTARY_CANAL","PEPTIC_ULCER","GASTROESOPHAGEA_REFLUX","IRRITABLE_BOWEL_SND","PSYCHOSIS","DEPRESSION","MANIC_DEPRESSION","POSTPARTUM_DEPRESSION","OBSESSIVE_COMPULSIVE_DIS","ALCOHOLISM_DRUG_ABUSE","SCHIZOPHRENIA","NERVOUS_SYSTEM","EPILEPSY","HEMICRANIA","MULTIPLE_SCLEROSIS","PARKISON","DEMENTIA","LIVER_GALL_STONE","KIDNEY_STONE","RENAL_FAILURE","VERTIGO","CARCINOMA_IN_SITU","LIVER_CA","LUNG_CA","BREAST_CA","GASTRIC_CA","COLORECTAL_CA","NASOPHARYNGEAL_CA","PROSTATE_CA","OTHER_CA","CATARACT","GLAUCOMA","XEROPHTHALMIA","RENTINAL_DETACHMENT","FLOATERS","BLIND","COLOR_BLIND","MENSTRURATION_REGULAR","MENSTRURATION_CURR","PREGNANCY","BIRTH","ELDERL_PRIMIGRAVIDA","ABORTION","BREAST_MILK","SEVERE_DYSMENORRHEA","MYOMA","OVARIAN_CYST","ENDOMETRIOSIS","CERVICAL_POLYP","UTERINE_CA","CERVICAL_CA","OVARIAN_CA","HBA1C","ANTI_HCV_AB_1","HBSAG_1","HBEAG_1","ANTI_HBS_AB_1","ANTI_HBC_AB_1","rs35645182_cate","rs34782958_cate","rs2198891_cate","rs190752175_cate","rs4712614_cate","rs35257346_cate","rs1871859_cate","rs635965_cate","rs867023908_cate"
    ], axis=1)
    return re_cate_df

def ectract_num(input_df):
    #feature select
    re_num_df = input_df.filter(items=["AGE","MENARCHE","MC_DURATION","BIRTH_TIMES","BODY_HEIGHT","BODY_WEIGHT","BMI","BODY_FAT_RATE","BODY_WAISTLINE","BODY_BUTTOCKS","WHR","SIT_SYSTOLIC_PRESSURE_AVERAGE","SIT_DIASTOLIC_PRESSURE_AVERAGE","SIT_HEARTBEAT_SPEED_AVERAGE","RBC","WBC","PLATELET","HB","HCT","FASTING_GLUCOSE","T_CHO","TG","HDL_C","LDL_C","T_BILIRUBIN","ALBUMIN","SGOT","SGPT","GAMMA_GT","AFP","BUN","CREATININE","URIC_ACID","MICROALB","TPMI_Osteoporosis_osteopenia and pathological fracture","TPMI_Osteoporosis","TPMI_Osteoporosis NOS","TPMI_Senile osteoporosis"
    ], axis=1)
    return re_num_df


In [20]:
cate_train_df = ectract_cate(train_df)
num_train_df = ectract_num(train_df)

cate_test_df = ectract_cate(test_df)
num_test_df = ectract_num(test_df)

print(cate_train_df.shape)
print(num_train_df.shape)

#針對SNP category類別，僅使用train類別資料建立binary encode編碼規則，再套用到test類別資料
from category_encoders import BinaryEncoder

binary_cols = [
    col for col in cate_train_df.columns
    if col.endswith("_cate")
]
print(f"binary_cols 筆數: {len(binary_cols)}")
encoder = BinaryEncoder(cols=binary_cols)
train_encode_df = encoder.fit_transform(cate_train_df)
test_encode_df = encoder.transform(cate_test_df)

cate_train_array = np.array(train_encode_df, dtype='float32')
cate_test_array = np.array(test_encode_df, dtype='float32')

joblib.dump(encoder, "binary_encoder.pkl")

#僅使用train數值資料建立正規化規則，再套用到test數值資料，回傳資料型態為numpy array
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_train_array = scaler.fit_transform(num_train_df)

num_test_array = scaler.transform(num_test_df)

joblib.dump(scaler, "snp_oste_scaler.pkl")


(52449, 89)
(52449, 38)
binary_cols 筆數: 9


['snp_oste_scaler.pkl']

In [21]:
#合併cate和num資料
train_array = np.concatenate([cate_train_array, num_train_array], axis=1)
train_os_array = train_df['OS'].to_numpy()

train_pos_array = train_array[train_os_array == '1']
train_neg_array = train_array[train_os_array == '0']
train_mid_array = train_array[train_os_array == '2']
print(train_pos_array.shape)
print(train_neg_array.shape)
print(train_mid_array.shape)


(4007, 143)
(34544, 143)
(13898, 143)


In [22]:
# training neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(train_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(train_neg_array, [median_center]).flatten()
#print(distances)

ext_train_neg_array = np.empty((1, train_neg_array.shape[1]), dtype=np.float32)

K = len(train_pos_array)

idx = np.argsort(distances)[-K:]
ext_train_neg_array = train_neg_array[idx]

print("POS:", train_pos_array.shape)
print("NEG:", ext_train_neg_array.shape)

POS: (4007, 143)
NEG: (4007, 143)


In [23]:
# training mid data
# 用POS和NEG取MID

ext_train_mid_array = np.empty((1, train_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(train_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_train_neg_array, axis=0)

pos_distances = euclidean_distances(train_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(train_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 7.387

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_train_mid_array = train_mid_array[mask]

print("mid:", ext_train_mid_array.shape)

mid: (4008, 143)


In [24]:
test_array = np.concatenate([cate_test_array, num_test_array], axis=1)
test_os_array = test_df['OS'].to_numpy()

test_pos_array = test_array[test_os_array == '1']
test_neg_array = test_array[test_os_array == '0']
test_mid_array = test_array[test_os_array == '2']
print(test_pos_array.shape)
print(test_neg_array.shape)
print(test_mid_array.shape)

(971, 143)
(8688, 143)
(3454, 143)


In [25]:
# testing neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(test_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(test_neg_array, [median_center]).flatten()
#print(distances)

ext_test_neg_array = np.empty((1, test_neg_array.shape[1]), dtype=np.float32)

K = len(test_pos_array)

idx = np.argsort(distances)[-K:]
ext_test_neg_array = test_neg_array[idx]

print("POS:", test_pos_array.shape)
print("NEG:", ext_test_neg_array.shape)

POS: (971, 143)
NEG: (971, 143)


In [26]:
# testing mid data
# 用POS和NEG取MID

ext_test_mid_array = np.empty((1, test_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(test_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_test_neg_array, axis=0)

pos_distances = euclidean_distances(test_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(test_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 7.41

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_test_mid_array = test_mid_array[mask]

print("mid:", ext_test_mid_array.shape)

mid: (971, 143)


In [27]:
training_X = np.vstack([train_pos_array, ext_train_neg_array])
training_X = np.vstack([training_X, ext_train_mid_array])
training_Y = np.concatenate([np.ones(train_pos_array.shape[0]), np.zeros(ext_train_neg_array.shape[0]), np.full(ext_train_mid_array.shape[0], 2)])
print(training_X.shape)
print(training_Y.shape)

testing_X = np.vstack([test_pos_array, ext_test_neg_array])
testing_X = np.vstack([testing_X, ext_test_mid_array])
testing_Y = np.concatenate([np.ones(test_pos_array.shape[0]), np.zeros(ext_test_neg_array.shape[0]), np.full(ext_test_mid_array.shape[0], 2)])
print(testing_X.shape)
print(testing_Y.shape)

(12022, 143)
(12022,)
(2913, 143)
(2913,)


In [28]:
# 儲存training&testing numpy array
np.save("snp_training_X.npy", training_X)
np.save("snp_training_Y.npy", training_Y)
np.save("snp_testing_X.npy", testing_X)
np.save("snp_testing_Y.npy", testing_Y)

In [29]:
import numpy as np
training_X = np.load("snp_training_X.npy")
training_Y = np.load("snp_training_Y.npy")
testing_X = np.load("snp_testing_X.npy")
testing_Y = np.load("snp_testing_Y.npy")
print(training_X.shape)
print(testing_X.shape)

(12022, 143)
(2913, 143)


In [30]:
import collections
print(collections.Counter(training_Y))
print(collections.Counter(testing_Y))

Counter({np.float64(2.0): 4008, np.float64(1.0): 4007, np.float64(0.0): 4007})
Counter({np.float64(1.0): 971, np.float64(0.0): 971, np.float64(2.0): 971})


In [31]:
from sklearn.metrics import accuracy_score,f1_score,matthews_corrcoef,confusion_matrix
def evalution_metrics_multi(test_label, labels_score):
    accuracy = accuracy_score(test_label, labels_score.round())
    confusion = confusion_matrix(test_label, labels_score.round())
    print(confusion)
    f1 = f1_score(test_label, labels_score.round(), average="macro")
    mcc = matthews_corrcoef(test_label, labels_score.round())
    metrics = np.round([accuracy, f1, mcc],2)
    columns=['accuracy', 'f1', 'mcc']
    metrics_dict = dict(zip(columns,metrics))
    
    return(metrics_dict)

In [32]:
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_validate
import optuna

/home/wenzhi/anaconda3/envs/twb_analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
cate_column = train_encode_df.columns
num_column = num_train_df.columns
feature_list = np.concatenate([cate_column, num_column])
print(len(feature_list))

143


In [34]:
from sklearn.model_selection import KFold
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [35]:
from sklearn.inspection import permutation_importance

def get_permutation_importance(model_name, feature_list):
    result = permutation_importance(
        model_name, 
        testing_X, 
        testing_Y,
        scoring="f1_macro",
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    ) 
    
    feature_importance = pd.DataFrame({
        "feature": feature_list,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)
    #print(feature_importance)
    #print(feature_importance.head(20))

    # 合併 Binary Encoding 欄位
    feature_importance["feature"] = (
        feature_importance["feature"]
        .str.replace(r'_\d+$', '', regex=True)
    )

    # 將同一特徵的重要性加總
    feature_importance = (
        feature_importance
        .groupby("feature", as_index=False)
        .agg({
            "importance_mean": "sum",
            "importance_std": "sum"
        })
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )

    print(feature_importance.head(10))

    # 列出SNP相關feature的排名
    target_features = [
        "rs35645182_cate",
        "rs34782958_cate",
        "rs2198891_cate",
        "rs190752175_cate",
        "rs4712614_cate",
        "rs35257346_cate",
        "rs1871859_cate",
        "rs635965_cate",
        "rs867023908_cate",
        "TPMI_Osteoporosis_osteopenia and pathological fracture",
        "TPMI_Osteoporosis",
        "TPMI_Osteoporosis NOS",
        "TPMI_Senile osteoporosis"
    ]

    feature_importance["rank"] = feature_importance.index + 1

    print(
        feature_importance[
            feature_importance["feature"].isin(target_features)
            ][["rank", "feature", "importance_mean"]].sort_values("rank")
    )

In [36]:
def svm_cv(model, X_train, Y_train, cv):
    scores = cross_validate(
        model,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'f1_macro',
            'precision_macro',
            'recall_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )
    print('Accuracy:', scores['test_accuracy'])
    print('Precision:', scores['test_precision_macro'])
    print('Recall:', scores['test_recall_macro'])
    print('F1 score', scores['test_f1_macro'])
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'])

    print('Accuracy:', scores['test_accuracy'].mean())
    print('Precision:', scores['test_precision_macro'].mean())
    print('Recall:', scores['test_recall_macro'].mean())
    print('F1 score', scores['test_f1_macro'].mean())
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'].mean())



In [37]:
#SVM
from sklearn import svm
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
svc = svm.SVC(probability=True)
#svm_cv(svc, training_X, training_Y, cv)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(svc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"snp_svc_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

svc_fit = svc.fit(training_X, training_Y)
svm_score = svc_fit.predict(testing_X)

print(classification_report(testing_Y.astype(float), svm_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), svm_score.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7367 ± 0.0064
precision_macro: 0.7387 ± 0.0067
recall_macro: 0.7367 ± 0.0068
f1_macro: 0.7360 ± 0.0070
matthews_corrcoef: 0.6063 ± 0.0094
              precision    recall  f1-score   support

         0.0       0.76      0.85      0.81       971
         1.0       0.82      0.74      0.78       971
         2.0       0.64      0.63      0.63       971

    accuracy                           0.74      2913
   macro avg       0.74      0.74      0.74      2913
weighted avg       0.74      0.74      0.74      2913

[[828   2 141]
 [ 46 719 206]
 [211 153 607]]


{'accuracy': np.float64(0.74), 'f1': np.float64(0.74), 'mcc': np.float64(0.61)}

In [38]:
get_permutation_importance(svc_fit, feature_list)


           feature  importance_mean  importance_std
0              AGE         0.087081        0.004908
1  FASTING_GLUCOSE         0.019490        0.002543
2               HB         0.014001        0.002437
3      BODY_WEIGHT         0.013279        0.003804
4              BMI         0.012558        0.003611
5             SGPT         0.012052        0.003903
6               TG         0.011942        0.002515
7      T_BILIRUBIN         0.011912        0.002427
8              HCT         0.010753        0.002117
9         GAMMA_GT         0.009161        0.002976
     rank                                            feature  importance_mean
14     15  TPMI_Osteoporosis_osteopenia and pathological ...         0.005082
21     22                           TPMI_Senile osteoporosis         0.002616
27     28                                   rs190752175_cate         0.001073
53     54                                   rs867023908_cate         0.000064
56     57                             

In [39]:
from sklearn.ensemble import RandomForestClassifier
def rf_objective(trial, X_train, Y_train, cv):

    rf = RandomForestClassifier(
        n_estimators = trial.suggest_int("n_estimators", 10, 200),
        max_depth = trial.suggest_int("max_depth", 3, 30),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state = 42,
        n_jobs = -1
    )

    scores = cross_validate(
        rf,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'precision_macro',
            'recall_macro',
            'f1_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [40]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: rf_objective(trial, training_X, training_Y, cv), n_trials=200)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_forest = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_forest)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"snp_rf_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

forest_fit = best_forest.fit(training_X, training_Y)
rf_score = forest_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), rf_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), rf_score.astype(float))

[I 2026-11-18 16:08:08,400] A new study created in memory with name: no-name-f78c06e5-b95b-4e2e-a88c-0efef1ea568c


[I 2026-11-18 16:08:09,876] Trial 0 finished with value: 0.6749705967817964 and parameters: {'n_estimators': 179, 'max_depth': 29, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.6749705967817964.
[I 2026-11-18 16:08:11,246] Trial 1 finished with value: 0.6513575132078822 and parameters: {'n_estimators': 158, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6749705967817964.
[I 2026-11-18 16:08:12,546] Trial 2 finished with value: 0.6351128313261054 and parameters: {'n_estimators': 156, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6749705967817964.
[I 2026-11-18 16:08:14,210] Trial 3 finished with value: 0.5908421898112912 and parameters: {'n_estimators': 197, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.67497059678179

{'n_estimators': 163, 'max_depth': 29, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best F1 Macro: 0.6913
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.6906 ± 0.0043
precision_macro: 0.6928 ± 0.0041
recall_macro: 0.6907 ± 0.0045
f1_macro: 0.6913 ± 0.0039
matthews_corrcoef: 0.5362 ± 0.0064
              precision    recall  f1-score   support

         0.0       0.75      0.75      0.75       971
         1.0       0.75      0.72      0.73       971
         2.0       0.55      0.57      0.56       971

    accuracy                           0.68      2913
   macro avg       0.68      0.68      0.68      2913
weighted avg       0.68      0.68      0.68      2913

[[732  11 228]
 [ 55 699 217]
 [192 225 554]]


{'accuracy': np.float64(0.68), 'f1': np.float64(0.68), 'mcc': np.float64(0.52)}

In [41]:
get_permutation_importance(forest_fit, feature_list)

              feature  importance_mean  importance_std
0                 AGE         0.016648        0.005382
1  MENSTRURATION_CURR         0.013431        0.004648
2                 HCT         0.006121        0.002082
3                SGPT         0.005179        0.003289
4                  HB         0.003966        0.001996
5     FASTING_GLUCOSE         0.003845        0.002166
6            GAMMA_GT         0.003791        0.001818
7                 RBC         0.003751        0.002617
8                  TG         0.003423        0.001627
9               HBA1C         0.002785        0.002630
     rank                                            feature  importance_mean
13     14  TPMI_Osteoporosis_osteopenia and pathological ...         0.002084
15     16                              TPMI_Osteoporosis NOS         0.001768
16     17                                   rs867023908_cate         0.001659
17     18                                  TPMI_Osteoporosis         0.001535
24   

In [42]:
# RF feature importance
feature_importance = pd.DataFrame({
    "feature": feature_list,
    "importance": best_forest.feature_importances_
}).sort_values("importance", ascending=False)

# 合併 Binary Encoding 欄位
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace(r'_\d+$', '', regex=True)
)

# 將同一特徵的重要性加總
feature_importance = (
    feature_importance
    .groupby("feature", as_index=False)
    .agg({
        "importance": "sum"
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance.head(10))

              feature  importance
0                 AGE    0.067765
1  MENSTRURATION_CURR    0.061547
2         BODY_WEIGHT    0.054682
3                 BMI    0.045555
4       BODY_FAT_RATE    0.040822
5       BODY_BUTTOCKS    0.034348
6         MC_DURATION    0.029013
7      BODY_WAISTLINE    0.025912
8                SGPT    0.025633
9            PLATELET    0.022458


In [43]:

from xgboost import XGBClassifier

def xgb_objective(trial, X_train, Y_train, cv):
    xgb = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        objective="multi:softprob",   # 多分類
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
    scores = cross_validate(
        xgb,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )
    
    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [44]:
# 建立 XGBClassifier 模型
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: xgb_objective(
        trial,
        training_X,
        training_Y,
        cv
    ),
    n_trials=200
)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_xgb = XGBClassifier(
    **study.best_params,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_xgb)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"snp_xgb_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

xgb_fit = best_xgb.fit(training_X, training_Y)
xgb_score = xgb_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), xgb_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), xgb_score.astype(float))

[I 2026-11-18 16:14:13,132] A new study created in memory with name: no-name-f12e18ae-f7e8-4db3-9875-f74862e87c9b


[I 2026-11-18 16:14:22,259] Trial 0 finished with value: 0.6672994218565864 and parameters: {'n_estimators': 148, 'max_depth': 10, 'learning_rate': 0.0010598635957110353, 'subsample': 0.7640213454470961, 'colsample_bytree': 0.6097384297997218, 'min_child_weight': 8, 'gamma': 0.5098872725927839, 'reg_alpha': 6.331465307852423e-07, 'reg_lambda': 2.3412982032291198e-05}. Best is trial 0 with value: 0.6672994218565864.
[I 2026-11-18 16:14:26,469] Trial 1 finished with value: 0.6562519635320605 and parameters: {'n_estimators': 72, 'max_depth': 7, 'learning_rate': 0.0042316838153732455, 'subsample': 0.9372336795296425, 'colsample_bytree': 0.6136382375915308, 'min_child_weight': 2, 'gamma': 0.5544073132071575, 'reg_alpha': 0.014753118172439609, 'reg_lambda': 0.32385073432641387}. Best is trial 0 with value: 0.6672994218565864.
[I 2026-11-18 16:14:32,908] Trial 2 finished with value: 0.6693677131861495 and parameters: {'n_estimators': 110, 'max_depth': 10, 'learning_rate': 0.00663245099870675,

{'n_estimators': 196, 'max_depth': 8, 'learning_rate': 0.10592307023547233, 'subsample': 0.6831131311189682, 'colsample_bytree': 0.8863491670470555, 'min_child_weight': 10, 'gamma': 1.1276912814997244, 'reg_alpha': 0.0003147538064098088, 'reg_lambda': 0.0005406542399574409}
Best F1 Macro: 0.7138
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7142 ± 0.0027
precision_macro: 0.7138 ± 0.0031
recall_macro: 0.7142 ± 0.0032
f1_macro: 0.7138 ± 0.0031
matthews_corrcoef: 0.5714 ± 0.0040
              precision    recall  f1-score   support

         0.0       0.78      0.79      0.78       971
         1.0       0.77      0.76      0.76       971
         2.0       0.61      0.61      0.61       971

    accuracy                           0.72      2913
   macro avg       0.72      0.72      0.72      2913
weighted avg       0.72      0.72      0.72      2913

[[768  19 184]
 [ 39 734 198]
 [181 197 593]]


{'accuracy': np.float64(0.72), 'f1': np.float64(0.72), 'mcc': np.float64(0.58)}

In [45]:
get_permutation_importance(xgb_fit, feature_list)

              feature  importance_mean  importance_std
0                 AGE         0.055265        0.005107
1         BODY_WEIGHT         0.026025        0.003979
2                 BMI         0.017193        0.004059
3     FASTING_GLUCOSE         0.016200        0.002896
4                SGPT         0.014944        0.002976
5         T_BILIRUBIN         0.012926        0.001999
6                 HCT         0.011005        0.002100
7            GAMMA_GT         0.010423        0.002086
8           URIC_ACID         0.009837        0.002540
9  MENSTRURATION_CURR         0.009187        0.003588
    rank                                            feature  importance_mean
11    12  TPMI_Osteoporosis_osteopenia and pathological ...         0.008869
30    31                                  TPMI_Osteoporosis         0.004837
34    35                                    rs35645182_cate         0.004212
36    37                                     rs1871859_cate         0.003492
38    39  

In [46]:
# XGBoost feature importance
feature_importance = pd.DataFrame({
    "feature": feature_list,
    "importance": best_xgb.feature_importances_
}).sort_values("importance", ascending=False)

# 合併 Binary Encoding 欄位
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace(r'_\d+$', '', regex=True)
)

# 將同一特徵的重要性加總
feature_importance = (
    feature_importance
    .groupby("feature", as_index=False)
    .agg({
        "importance": "sum"
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print(feature_importance.head(10))

              feature  importance
0  MENSTRURATION_CURR    0.181872
1                 AGE    0.026680
2         BODY_WEIGHT    0.021882
3                 BMI    0.019074
4     rs35645182_cate    0.016204
5       rs635965_cate    0.016097
6      rs4712614_cate    0.015109
7               BIRTH    0.014113
8     rs35257346_cate    0.013671
9      rs1871859_cate    0.013566


In [47]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
def hgbc_objective(trial, X_train, Y_train, cv):

    hgbc = HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 100),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 100),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-8, 10.0, log=True),
        max_bins=trial.suggest_int("max_bins", 64, 255),
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=10,
        random_state=42
    )

    scores = cross_validate(
        hgbc,
        X_train,
        Y_train,
        cv = cv,
        scoring=[
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [48]:
from sklearn.ensemble import HistGradientBoostingClassifier
study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: hgbc_objective(trial, training_X, training_Y, cv),
    n_trials=200
)
best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_hgbc = HistGradientBoostingClassifier(
    **study.best_params,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=42
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_hgbc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"snp_hgbc_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

hgbc_fit = best_hgbc.fit(training_X, training_Y)
hgbc_score = hgbc_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), hgbc_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), hgbc_score.astype(float))

[I 2026-11-18 16:30:26,302] A new study created in memory with name: no-name-1bb76f68-6ac4-45a5-a4b3-f1514431f034
[I 2026-11-18 16:30:30,953] Trial 0 finished with value: 0.6534856998057202 and parameters: {'learning_rate': 0.0076278887935379396, 'max_iter': 81, 'max_depth': 7, 'max_leaf_nodes': 65, 'min_samples_leaf': 58, 'l2_regularization': 1.0285570261249626e-07, 'max_bins': 131}. Best is trial 0 with value: 0.6534856998057202.
[I 2026-11-18 16:30:35,514] Trial 1 finished with value: 0.6798842264518596 and parameters: {'learning_rate': 0.024148982629777535, 'max_iter': 75, 'max_depth': 9, 'max_leaf_nodes': 100, 'min_samples_leaf': 61, 'l2_regularization': 0.8502952130378874, 'max_bins': 147}. Best is trial 1 with value: 0.6798842264518596.
[I 2026-11-18 16:30:43,703] Trial 2 finished with value: 0.697509190697246 and parameters: {'learning_rate': 0.04340547946019583, 'max_iter': 146, 'max_depth': 11, 'max_leaf_nodes': 61, 'min_samples_leaf': 93, 'l2_regularization': 4.2353398001987

{'learning_rate': 0.03530280097659951, 'max_iter': 185, 'max_depth': 19, 'max_leaf_nodes': 31, 'min_samples_leaf': 10, 'l2_regularization': 0.0018254429898209748, 'max_bins': 154}
Best F1 Macro: 0.7152
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7151 ± 0.0036
precision_macro: 0.7158 ± 0.0034
recall_macro: 0.7151 ± 0.0042
f1_macro: 0.7152 ± 0.0038
matthews_corrcoef: 0.5729 ± 0.0054
              precision    recall  f1-score   support

         0.0       0.77      0.81      0.79       971
         1.0       0.77      0.74      0.75       971
         2.0       0.60      0.59      0.59       971

    accuracy                           0.71      2913
   macro avg       0.71      0.71      0.71      2913
weighted avg       0.71      0.71      0.71      2913

[[785  10 176]
 [ 42 721 208]
 [188 210 573]]


{'accuracy': np.float64(0.71), 'f1': np.float64(0.71), 'mcc': np.float64(0.57)}

In [49]:
get_permutation_importance(hgbc_fit, feature_list)

              feature  importance_mean  importance_std
0                 AGE         0.054520        0.005056
1         BODY_WEIGHT         0.027966        0.003701
2     FASTING_GLUCOSE         0.016697        0.002554
3  MENSTRURATION_CURR         0.015254        0.004320
4                SGPT         0.013044        0.003556
5                 BMI         0.010419        0.003499
6                  TG         0.009516        0.002361
7                 RBC         0.008332        0.003559
8            GAMMA_GT         0.007406        0.002715
9         T_BILIRUBIN         0.007084        0.002004
     rank                                            feature  importance_mean
27     28                           TPMI_Senile osteoporosis         0.001178
53     54  TPMI_Osteoporosis_osteopenia and pathological ...         0.000266
76     77                              TPMI_Osteoporosis NOS         0.000000
87     88                                   rs867023908_cate        -0.000047
103  

In [50]:
import joblib

joblib.dump(svc, "snp_svm(3class).pkl")
joblib.dump(best_forest, "snp_rf(3class).pkl")
joblib.dump(best_xgb, "snp_xgboost(3class).pkl")
joblib.dump(best_hgbc, "snp_HGBC(3class).pkl")

['snp_HGBC(3class).pkl']

In [51]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    clf1 = joblib.load("snp_svc_fold_" + str(fold) + ".pkl")
    clf2 = joblib.load("snp_rf_fold_" + str(fold) + ".pkl")
    clf3 = joblib.load("snp_xgb_fold_" + str(fold) + ".pkl")
    clf4 = joblib.load("snp_hgbc_fold_" + str(fold) + ".pkl")

    # 建立新的模型
    model = VotingClassifier(
        estimators=[('svm', clf1), ('xgb', clf3), ('hgbc', clf4)],
        voting='soft'  # 'soft' for probability-based voting
    )

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"snp_ensemble_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )    

# Ensemble model (majority voting)
clf1 = joblib.load("snp_svm(3class).pkl")
clf2 = joblib.load("snp_rf(3class).pkl")
clf3 = joblib.load("snp_xgboost(3class).pkl")
clf4 = joblib.load("snp_HGBC(3class).pkl")
ensemble = VotingClassifier(
    estimators=[('svm', clf1), ('xgb', clf3), ('hgbc', clf4)],
    voting='soft'  # 'soft' for probability-based voting
)

# Train
ensemble.fit(training_X, training_Y)

# Predictions
y_pred = ensemble.predict(testing_X)

# Evaluate
print(classification_report(testing_Y.astype(float), y_pred.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), y_pred.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7270 ± 0.0055
precision_macro: 0.7273 ± 0.0043
recall_macro: 0.7270 ± 0.0062
f1_macro: 0.7268 ± 0.0053
matthews_corrcoef: 0.5908 ± 0.0081
              precision    recall  f1-score   support

         0.0       0.77      0.81      0.79       971
         1.0       0.79      0.75      0.77       971
         2.0       0.62      0.61      0.61       971

    accuracy                           0.73      2913
   macro avg       0.73      0.73      0.73      2913
weighted avg       0.73      0.73      0.73      2913

[[791   8 172]
 [ 42 731 198]
 [192 186 593]]


{'accuracy': np.float64(0.73), 'f1': np.float64(0.73), 'mcc': np.float64(0.59)}

In [52]:
get_permutation_importance(ensemble, feature_list)

           feature  importance_mean  importance_std
0              AGE         0.062285        0.005223
1  FASTING_GLUCOSE         0.017447        0.002251
2      BODY_WEIGHT         0.016911        0.003458
3             SGPT         0.010997        0.002952
4              HCT         0.007692        0.001929
5               TG         0.007260        0.002842
6      T_BILIRUBIN         0.006868        0.002105
7              BMI         0.006788        0.004064
8         GAMMA_GT         0.006539        0.002785
9         MICROALB         0.006414        0.002154
     rank                                            feature  importance_mean
14     15  TPMI_Osteoporosis_osteopenia and pathological ...         0.003904
50     51                                     rs4712614_cate         0.000318
55     56                           TPMI_Senile osteoporosis         0.000188
64     65                                   rs190752175_cate         0.000033
79     80                             

In [53]:
joblib.dump(ensemble, "snp_ensemble(3class).pkl")

['snp_ensemble(3class).pkl']